# 114 — Ciclo ReAct y observación del entorno

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**ReAct** (Yao et al., arXiv:2210.03629) estructura cada iteración del agente en tres
elementos: **thought** (razonamiento en texto, sin efectos), **action** (invocación de
una herramienta con argumentos, o `finish`) y **observation** (la respuesta REAL del
entorno — el modelo no la genera: la recibe).

```text
repetir hasta finish o presupuesto agotado:
    thought_t     = LLM(contexto)             # razonar sobre lo observado
    action_t      = LLM(contexto + thought)   # decidir tool + args
    observation_t = entorno.ejecutar(action)  # información nueva y verificada
    contexto     += thought + action + observation
```

La observación es la única entrada de información nueva al bucle: sin ella el modelo
solo puede alucinar el estado del mundo (modo chain-of-thought). Los errores también
son observaciones — y de las más valiosas para el siguiente thought.

### 🔄 Lo que garantiza el patrón (bien implementado)

1. **Grounding:** cada decisión se toma sobre el último estado observado.
2. **Traza auditable:** la secuencia `(thought, action, observation)*` explica el porqué
   de cada paso — depurar un agente es leer su traza.
3. **Parada:** por decisión (`finish` con condiciones verificadas) o por presupuesto de
   pasos; nunca por "ya ejecuté lo que tenía pensado".

El laboratorio `agent` emite una traza de dos acciones (`status()` → obs
`{"healthy": true}`, `sum(7,5)` → obs `12`) con los thoughts implícitos: la condición
de éxito (`healthy == true` y `sum == 12`) se evalúa contra observaciones.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Ejercicios

**Ejercicio 1 — Anatomía de la traza.** Ejecuta `run_lab("agent", seed=114)` y verifica
`kind` y `evidence`. Después recorre `result["trace"]` y clasifica cada campo de cada
paso como generado por la *política* (action) o por el *entorno* (observation). ¿Dónde
estarían los thoughts si esta traza fuera de un agente LLM con ReAct explícito?

**Ejercicio 2 — Escribe los thoughts.** La traza del laboratorio no incluye thoughts.
Escríbelos a mano: Thought 1 (antes de `status()`), Thought 2 (después de observar
`healthy: true`, antes de `sum`), Thought 3 (antes de `finish`). Cada thought debe
referirse SOLO a lo observado hasta ese momento, sin adelantar resultados.

**Ejercicio 3 — Traza contrafactual.** Escribe la traza completa
(thought → action → observation) si `status()` devolviera `{"healthy": false}` en el
paso 1 y `true` en un reintento. ¿En qué paso puede el agente invocar `finish` en éxito
y qué condición NO puede darse por verificada antes?

**Ejercicio 4 — CoT vs Act-only vs ReAct.** Para la pregunta "¿en qué año se fundó la
universidad donde estudió la autora de *Frankenstein*?", describe cómo fallaría (o no)
cada régimen: (a) chain-of-thought sin herramientas, (b) act-only con un buscador pero
sin razonamiento intermedio, (c) ReAct. Señala el punto exacto donde (a) puede alucinar
y donde (b) puede perderse.

In [ ]:
# TODO: ejecuta run_lab("agent", seed=114)
# TODO: comprueba que el resultado incluya las claves 'kind' y 'evidence'
result = None


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicio 1: clasifica los campos de la traza
result = run_lab("agent", seed=114)
for paso in result["result"]["trace"]:
    accion = paso["action"]        # ¿quién genera esto: política o entorno?
    obs = paso["observation"]      # ¿y esto?
    # imprime cada campo con su origen ("politica" / "entorno")
    pass


In [ ]:
# TU RESPUESTA AQUÍ — Ejercicios 2 y 3: thoughts y traza contrafactual
thoughts = [
    # "Thought 1: ...",
    # "Thought 2: ...",
    # "Thought 3: ...",
]
traza_contrafactual = [
    # {"thought": "...", "action": {...}, "observation": {...}},
]


## Reflexión

1. En la traza del laboratorio, ¿qué campo de cada paso proviene del entorno y cuál
   proviene de la política de decisión? ¿Por qué el grounding desaparecería si el mismo
   componente pudiera escribir ambos?
2. ReAct reduce la alucinación frente a chain-of-thought puro, pero el paper reporta
   errores de *reasoning* aun con observaciones correctas. ¿Qué aspecto de un thought
   temprano puede sesgar toda la trayectoria y qué mecanismo de la clase 115 lo mitiga?
3. Si `sum(7, 5)` devolviera `{"error": "timeout"}`, ¿qué debería contener el siguiente
   thought y por qué silenciar ese error en la herramienta "ciega" al agente?